# VIO first-light -- coordinator capture review

First look at the on-vehicle **disparity + still** captures from the first VIO-in-the-loop
flight (`260709-vio-first-light`, 2026-07-09; fables PR #10, coordinator #64).

Each frame has a JSON sidecar (`kind` disparity/still, `wall_clock_unix`, **`monotonic_ns`**,
`sensor_timestamp_ns`, `device_seq`, dims, and stills' `exposure_us`/`iso`). Timing uses
**`monotonic_ns`** -- the coordinator has no RTC, so `wall_clock` steps when NTP locks; the
monotonic clock is the one that TIMESYNC-aligns to the FC (`vio-quality-experiments.md` E13).

Logic lives in `analysis/coordinator_captures.py`; this notebook is a thin, papermill-driven view.

In [3]:
import sys, os, json
import matplotlib.pyplot as plt
import numpy as np
# works whether the kernel cwd is the repo root or analysis/
for _p in (os.getcwd(), os.path.join(os.getcwd(), "analysis")):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import coordinator_captures as cc

ModuleNotFoundError: No module named 'analysis'

In [ ]:
# papermill parameters
FLIGHT_DIR = "/home/jovyan/datasets/flights/rekon10/260709-vio-first-light"
SESSION_DIR = FLIGHT_DIR + "/captures/ed14fd493245/20260710T003719Z"
OUT_DIR = FLIGHT_DIR + "/derived"
ANIM_FPS = 15
MAX_ANIM_FRAMES = 0

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)
df = cc.index_session(SESSION_DIR) if SESSION_DIR else cc.index_flight(FLIGHT_DIR)
print(json.dumps(cc.summarize(df), indent=2))
df.head()

## Capture cadence, exposure & the canopy light-drop

ISO / exposure climbing is the woods traverse (less light under canopy). Disparity ~1 Hz;
stills ~0.2 Hz. Use the `monotonic` timeline, not wall-clock.

In [ ]:
t0 = df["monotonic_ns"].min()
df = df.assign(t_rel=(df["monotonic_ns"] - t0) / 1e9)
st, dp = df[df.kind=="still"], df[df.kind=="disparity"]
fig, ax = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
ax[0].plot(st.t_rel, st.iso, ".-", ms=4); ax[0].set_ylabel("ISO (stills)"); ax[0].grid(alpha=.3)
ax[0].set_title(os.path.basename(FLIGHT_DIR))
ax[1].plot(st.t_rel, st.exposure_us/1000, ".-", ms=4, color="tab:orange")
ax[1].set_ylabel("exposure (ms)"); ax[1].grid(alpha=.3)
tt = dp.t_rel.to_numpy()
ax[2].plot(tt[1:], np.diff(tt), ".", ms=3, color="tab:green")
ax[2].set_ylabel("disp frame dt (s)"); ax[2].set_xlabel("t since session start (s)"); ax[2].grid(alpha=.3)
fig.tight_layout(); plt.show()

## Rest vs motion

The session opens with a **long true-rest period** (SSH bring-up before takeoff) -- a clean
stationary baseline for sensor-noise / EKF-at-rest characterization. The exact arm/takeoff
boundary comes from the FC `.bin` (join below); until decoded, exposure/disparity trends are the proxy.

In [ ]:
# disparity animation -> mp4 (set MAX_ANIM_FRAMES small for a quick preview)
d = df if not MAX_ANIM_FRAMES else df[df.kind=="disparity"].iloc[:MAX_ANIM_FRAMES]
mp4 = cc.animate_disparity(d, os.path.join(OUT_DIR, "disparity.mp4"), fps=ANIM_FPS)
print("wrote", mp4, os.path.getsize(mp4)//1024, "KB")
from IPython.display import Video
Video(mp4, embed=False, width=640)

## TODO -- next passes (see #64)

- **Join to the FC trajectory.** Align `monotonic_ns` to the FC EKF/GPS via TIMESYNC (E13,
  stable ~160 ppm) once the `260709` `.bin` is decoded -> tag each frame with pose / GPS-fix
  status; mark arm/takeoff and RTK-degradation windows.
- **Disparity overlays** (the interesting one): register successive disparity frames / overlay
  onto the RGB still or a common frame; per-frame valid-disparity fraction & near-range density
  as a *scene-difficulty* signal through the canopy.
- **Feature-vector capture** (not recorded this flight) unblocks offline VINS pose regen (#42).